# Study report: stimulus-sampling

The report template of the study. It reads one built instance of the
study (`scripts/benchmarks/study.py build`) and shows its central
results, its hypotheses with their checks, and interactive plots: hover
for values, drag to zoom, click a legend entry to hide a series, and use
the menu above a plot to select the metric.

Commit this file without outputs. The executed copy of an instance goes
into the folder of that instance.

In [ ]:
# Parameters: the instance folder. Set LATBENCH_INSTANCE, or change the default.
import os
import subprocess
import sys
from pathlib import Path

INSTANCE = Path(os.environ.get(
    "LATBENCH_INSTANCE",
    "/wrk/z/lab-benchmarks/results/kernel-latency/st-measurement-validity/st-stimulus-sampling@nucleo_g474re"))
REPO = Path(subprocess.run(["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True,
                           check=True).stdout.strip())

In [ ]:
import hashlib

import pandas as pd
import plotly.graph_objects as go

sys.path.insert(0, str(REPO / "scripts" / "benchmarks"))
import data  # noqa: E402
import loadmodel  # noqa: E402

pd.set_option("display.max_colwidth", None)
inst = data.load_instance(INSTANCE)
s = inst.study
print(s["instance"], "| board", s["board"], "| tools", s["made_by"]["safety_rev"][:12])
for name, digest in s["design"]["files"].items():
    now = hashlib.sha256((inst.design_folder / name).read_bytes()).hexdigest()
    if now != digest:
        print(f"WARNING: {name} changed after the build of the instance")
print()
print(s["question"])

## Central results

In [ ]:
inst.central()

## Hypotheses

A hypothesis is one claim about a mechanism. It is wrong if one of its
checks is wrong. The statements and the limits were written before the
runs; the column `basis` says where each number comes from.

In [ ]:
inst.hypotheses()

In [ ]:
inst.checks()

## Tick phase coverage

The share of the edges in each of the 64 phase bins of the tick, against
the uniform share (1 = even). `random` is the stimulus of all
experiments; `fixed-scan` moves the phase by 1 cycle per edge.

In [ ]:
fig = go.Figure()
for exp in ("irq-phase-1m", "wake-phase-1m"):
    for run, tree, point, rep in inst.points(exp):
        for metric, ph in rep["phases"].items():
            tot = sum(b[1] for b in ph["bins"])
            fig.add_trace(go.Bar(x=[b[0] for b in ph["bins"]], y=[b[1] * len(ph["bins"]) / tot for b in ph["bins"]],
                                 name=f"{exp} {point} {metric}"))
fig.update_layout(barmode="group", xaxis_title="phase bin (of 64)", yaxis_title="share / uniform share", height=450)
fig.show()

In [ ]:
def ccdf_figure(series, title, metrics):
    """series: {label: {metric: hist}}. A menu selects the metric."""
    fig = go.Figure()
    for i, metric in enumerate(metrics):
        for label, hists in series.items():
            x, y = data.ccdf(hists[metric])
            fig.add_trace(go.Scatter(x=x, y=y, mode="lines", line_shape="hv", name=label, visible=(i == 0),
                                     hovertemplate="%{x} cycles: %{y:.2e}<extra>" + label + "</extra>"))
    k = len(series)
    fig.update_layout(
        title=title, xaxis_title="latency (cycles)", yaxis_title="share of samples >= x", yaxis_type="log", height=500,
        updatemenus=[{"buttons": [{"label": m, "method": "update",
                                   "args": [{"visible": [j // k == i for j in range(k * len(metrics))]}]}
                                  for i, m in enumerate(metrics)], "x": 1.0, "y": 1.15}])
    return fig

## CCDF by stimulus seed (stim-seed-1m)

Eight stimulus seeds at the same point, with the night run of
`wp3-wake-1m` `baseline-t64` (a reference run of this instance) dashed.

In [ ]:
series = {point: rep["hists"] for _, _, point, rep in inst.points("stim-seed-1m")}
ref = next(r for r in inst.instance["references"] if r["id"].startswith("wp3-wake-1m/"))
series["night baseline-t64"] = \
    next(p["reps"][0]["hists"] for p in inst.app_data(ref["id"])["points"] if p["point"] == "baseline-t64")
fig = ccdf_figure(series, "stim-seed-1m: CCDF by stimulus seed", ["irq", "isr2thr", "irq2thr"])
for t in fig.data:
    if t.name == "night baseline-t64":
        t.line.dash = "dash"
fig.show()

## CCDF by load seed (load-seed-tail)

The 4M run and three load seeds of 1M samples. The CCDF is a share, so
runs of different length are comparable.

In [ ]:
lst = inst.points("load-seed-tail")
ccdf_figure({point: rep["hists"] for _, _, point, rep in lst}, "load-seed-tail: CCDF by load seed",
            ["irq", "isr2thr", "irq2thr"]).show()

## Tail count against the ticks with many expiries

H3 a..c predicted that the count of samples >= 4288 cycles (per 1M
samples) grows with the ticks with >= 3 load timer expiries (from
`loadmodel.py`). The dashed line is that prediction, through the mean of
the 8 stimulus seeds.

In [ ]:
THRESHOLD, K = 4288, 3


def tail_row(label, rep):
    """Ticks with >= K expiries (model) and samples >= THRESHOLD, both per 1M samples."""
    params = loadmodel.log_params({k: str(v) for k, v in rep["meta"].items()})
    ticks = int(params["seconds"] * params["ticks_per_sec"]) + 1
    tset = loadmodel.timers(params["seed"], params["timers"], params["min_ms"], params["max_ms"],
                            params["ticks_per_sec"])
    s = loadmodel.summary(loadmodel.expiries_per_tick(tset, ticks), 1)
    h = rep["hists"]["irq"]
    count = sum(c for v, c in h["bins"] if v >= THRESHOLD) + sum(c for v, c, _ in h["coarse"] if v >= THRESHOLD)
    per_m = 1e6 / h["n"]
    return {"point": label, "load_seed": f"0x{params['seed']:08x}",
            "ticks": loadmodel.at_least(s["counts"], K) * per_m, "samples": count * per_m}


df = pd.DataFrame([tail_row(point, rep) for _, _, point, rep in lst])
seeds = pd.DataFrame([tail_row(point, rep) for _, _, point, rep in inst.points("stim-seed-1m")])
x0, y0 = seeds.ticks.mean(), seeds.samples.mean()
fig = go.Figure(go.Scatter(x=df.ticks, y=df.samples, mode="markers+text", text=df.point,
                           textposition="top center", marker_size=10, name="load-seed-tail"))
fig.add_trace(go.Scatter(x=[x0], y=[y0], mode="markers", marker_symbol="diamond", marker_size=10,
                         error_y={"type": "data", "symmetric": False, "array": [seeds.samples.max() - y0],
                                  "arrayminus": [y0 - seeds.samples.min()]},
                         name="stim-seed-1m: mean and range of 8 seeds"))
xs = [0, df.ticks.max() * 1.1]
fig.add_trace(go.Scatter(x=xs, y=[x * y0 / x0 for x in xs], mode="lines", line_dash="dash",
                         name="proportional to the ticks (H3)"))
fig.update_layout(xaxis_title=f"ticks with >= {K} expiries per 1M samples (loadmodel.py)",
                  yaxis_title=f"samples >= {THRESHOLD} cycles per 1M", height=450)
fig.show()
df